In [14]:
from ipywidgets import HBox, VBox, Box, fixed, interactive_output
from plotly.graph_objects import FigureWidget
from IPython.display import display
from pycaret.classification import setup, compare_models, plot_model

from django.db import connection
from detections.models import Detection

import pandas as pd
import plotly.express as px
import sys
import os
import csv

os.environ["DJANGO_ALLOW_ASYNC_UNSAFE"] = "true"
sys.path.append(os.path.abspath("./"))
import helpers as _

In [15]:
detections_query =  'SELECT d.id, ' + \
                    'STRFTIME("%Y-%m-%d %H:%M:%S", c.date) AS datetime, ' + \
                    'STRFTIME("%H", c.date) AS hour, ' + \
                    'STRFTIME("%Y%m%d%H", c.date) AS datehour_key, ' +  \
                       'z.id AS zone_id, ' +  \
                       'z.name AS zone_name, ' +  \
                       'f.[index] AS class_index, ' +  \
                       'f.name AS class_name, ' +  \
                       'd.center_x AS center_x_norm, ' +  \
                       'd.center_y AS center_y_norm ' +  \
                  'FROM detections_detection d ' +  \
                       'LEFT JOIN ' +  \
                       'detections_capture c ON d.capture_id = c.id ' +  \
                       'LEFT JOIN ' +  \
                       'configuration_family f ON d.family_id = f.id ' +  \
                       'LEFT JOIN ' +  \
                       'configuration_zone z ON d.zone_id = z.id ' +  \
                 'WHERE c.status == "archived" AND  ' +  \
                       'c.source == "vision" AND  ' +  \
                       '(f.[index] == 1 OR  ' +  \
                        'f.[index] == 2)  ' +  \
                 'ORDER BY c.date ASC '
                        
detections = Detection.objects.raw(detections_query)

def save_rows(rows, query, path): 
    columns = list()
    
    with connection.cursor() as cursor:
        cursor.execute(query)
        columns = [col[0] for col in cursor.description]
    
    with open(path, 'w+', newline='') as file:
        writer = csv.writer(file)

        writer.writerow(columns)
        
        for obj in rows:
            row = list()
            
            for col in columns:
                row.append(obj.__dict__[col])
                
            writer.writerow(row)
    
        print(f"{len(rows)} lignes exportées")

save_rows(detections, detections_query, 'Followchon/data/detections.csv')

56896 lignes exportées
